# DNAK: reproducing the main claims in Google Colab

Paper: *DNAK: Exact Acceleration of Lloyd's k-Means via Delaunay-Neighbor Pruning* (B. Kr. Paul, K. Giri).
Code: https://github.com/barunkrpaul/dnak-kmeans

Run the cells in order (Runtime > Run all). The notebook checks

1. exactness: every method returns the same clustering as Lloyd's algorithm;
2. the seven-method comparison on a reduced problem;
3. the trend with k in the plane (DNAK against Exponion).

Colab CPUs differ between sessions, so absolute times will differ from the paper. Problem sizes are reduced so that the notebook finishes in about 10 to 15 minutes. The full experiments are run with `experiments/run_all.sh`.

In [ ]:
!git clone -q https://github.com/barunkrpaul/dnak-kmeans.git
%cd dnak-kmeans
%pip -q install -e .
import sys; sys.path.insert(0, "src")

In [ ]:
# 1. Exactness: iterations, assignments and centroids identical to Lloyd's algorithm
from dnak_kmeans import (ALL_ALGOS, ALGOS, run_algo, is_exact, warmup,
                         make_blobs, kmeanspp_init)
print("compiling kernels (about one minute the first time) ...")
warmup()
n_checks = 0
for d in (2, 3, 4):
    for seed in range(3):
        X = make_blobs(4000, d, 24, seed)
        C0 = kmeanspp_init(X, 30, seed + 50)
        ref = run_algo("Lloyd", X, C0)
        for name in ALL_ALGOS[1:]:
            assert is_exact(run_algo(name, X, C0), ref), (name, d, seed)
            n_checks += 1
print(f"PASS: {n_checks} runs identical to Lloyd's algorithm")

In [ ]:
# 2. Seven-method comparison (reduced size: n = 50,000, d = 2, k = 128)
import numpy as np
X = make_blobs(50000, 2, 128, seed=23)
times = {m: [] for m in ALGOS}; comps = {m: [] for m in ALGOS}
for seed in (101, 202, 303):
    C0 = kmeanspp_init(X, 128, seed)
    ref = run_algo("Lloyd", X, C0)
    for m in ALGOS:
        r = ref if m == "Lloyd" else run_algo(m, X, C0)
        assert is_exact(r, ref)
        times[m].append(r["time"]); comps[m].append(r["comps"])
print(f"{'method':10s} {'speed-up':>9s} {'DC reduction':>13s}")
for m in ALGOS:
    sp = np.mean(np.array(times["Lloyd"]) / np.array(times[m]))
    dc = np.mean(np.array(comps["Lloyd"]) / np.array(comps[m], float))
    print(f"{m:10s} {sp:8.1f}x {dc:12.1f}x")

In [ ]:
# 3. DNAK against Exponion as k grows (d = 2, n = 50,000)
import matplotlib.pyplot as plt
KS = [64, 128, 256, 512]
res = {"Exponion": [], "DNAK": []}
for k in KS:
    X = make_blobs(50000, 2, k, seed=2000 + k)
    sp = {"Exponion": [], "DNAK": []}
    for seed in (101, 202):
        C0 = kmeanspp_init(X, k, seed)
        ref = run_algo("Lloyd", X, C0)
        for m in sp:
            r = run_algo(m, X, C0)
            assert is_exact(r, ref)
            sp[m].append(ref["time"] / r["time"])
    for m in sp:
        res[m].append(np.mean(sp[m]))
    print(f"k={k:4d}: Exponion {res['Exponion'][-1]:5.1f}x   DNAK {res['DNAK'][-1]:5.1f}x")
plt.figure(figsize=(5.5, 3.2), dpi=120)
plt.plot(KS, res["Exponion"], "o--", label="Exponion")
plt.plot(KS, res["DNAK"], "s-", label="DNAK")
plt.xscale("log", base=2); plt.xticks(KS, KS)
plt.xlabel("k"); plt.ylabel("speed-up over Lloyd"); plt.legend(frameon=False)
plt.title("d = 2, n = 50,000"); plt.tight_layout(); plt.show()
print("Paper (n = 100,000): Exponion ahead at k = 64, DNAK clearly ahead from k = 256.")